# 05 — Evaluation Analysis & Charts

Owner: **Hoàng Đức Kiên** (QA) — Task T10 / T19.

Goals:
- Load CF eval scores (`evaluation/cf_eval_scores.csv` or `reports/cf_eval_scores.csv`)
- Plot HR@10 / NDCG@10
- Spot-check 5 users' recommendations
- Export charts under `reports/charts/`

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
sns.set_theme(style="whitegrid")
CHARTS = ROOT / "reports" / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)
print("Setup OK")

## Load eval scores

In [ ]:
candidates = [
    ROOT / "evaluation" / "cf_eval_scores.csv",
    ROOT / "reports" / "cf_eval_scores.csv",
]
csv = next((p for p in candidates if p.exists()), None)
if csv is None:
    raise FileNotFoundError(
        "Run `python scripts/run_evaluation.py` first "
        "(or execute notebook 04 eval sweep)."
    )
df = pd.read_csv(csv)
print("loaded", csv)
df

## Chart 1 — HR@10 summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = df.copy()
if "sample" in plot_df.columns:
    sns.barplot(data=plot_df, x="sample", y="HR@10", ax=ax)
    ax.set_xlabel("sample size")
else:
    ax.bar(["HR@10"], [float(plot_df["HR@10"].iloc[0])])
ax.set_title("CF Hit Rate@10")
ax.set_ylabel("HR@10")
plt.tight_layout()
fig.savefig(CHARTS / "hr_vs_topk.png", dpi=120)
print("saved", CHARTS / "hr_vs_topk.png")
fig

## Chart 2 — NDCG@10 summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = df.copy()
if "sample" in plot_df.columns:
    sns.barplot(data=plot_df, x="sample", y="NDCG@10", ax=ax)
    ax.set_xlabel("sample size")
else:
    ax.bar(["NDCG@10"], [float(plot_df["NDCG@10"].iloc[0])])
ax.set_title("CF NDCG@10")
ax.set_ylabel("NDCG@10")
plt.tight_layout()
fig.savefig(CHARTS / "ndcg_vs_topk.png", dpi=120)
print("saved", CHARTS / "ndcg_vs_topk.png")
fig

## Spot-check 5 users

Verify recommendations exclude already-rated movies.

In [ ]:
from recommender_cf import load_cf_artifacts, recommend_for_user
from data_processing import load_processed

cf = load_cf_artifacts()
movies, ratings_cf, _ = load_processed()

rows = []
for uid in [1, 42, 100, 500, 1000]:
    try:
        recs = recommend_for_user(cf, movies, uid, top_k=5)
        seen = set(ratings_cf.loc[ratings_cf.userId == uid, "movieId"])
        leak = set(recs.movieId) & seen
        print(f"--- user {uid} --- leak={len(leak)}")
        print(recs[["title", "score"]].to_string(index=False))
        assert not leak, f"seen-movie leak for user {uid}"
        rows.append({"userId": uid, "n_recs": len(recs), "leak": 0})
    except (KeyError, ValueError) as exc:
        print(f"user {uid}: {exc}")
        rows.append({"userId": uid, "n_recs": 0, "leak": None})
pd.DataFrame(rows)